In [1]:
import sys
from pathlib import Path
import pandas as pd
import html
import bm25s
import Stemmer
import json
from datasets import load_dataset

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from eval.evaluation import (
    gold_converter, write_predictions, run_official_eval,
    score_answers, score_retrieval, exact_match_score, f1_score,
)

In [2]:
CORPUS_PATH = ROOT / "data" / "corpus" / "pooled_corpus.parquet"
BM25_DIR = ROOT / "data" / "indices" / "bm25"
BM25_DIR.mkdir(parents=True, exist_ok=True)

RUNS_DIR = ROOT / "data" / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR = ROOT / "splits"
GOLD_DIR = ROOT / "results" / "gold"

# load corpus
corpus = pd.read_parquet(CORPUS_PATH, columns=["pid", "title", "text"])

# BM25 settings: used for the corpus, the queries and the saved config
STOPWORDS = "en"
STEMMER_LANG = "english"
K1 = 1.5
B = 0.75
METHOD = "lucene"

# setup stemmer
stemmer = Stemmer.Stemmer(STEMMER_LANG)

In [3]:
# combine title with text (titles are informative for entity names)
corpus_texts = [html.unescape(title) + " " + text for title, text in zip(corpus.title, corpus.text)]

# tokenize with stemming
corpus_tokens = bm25s.tokenize(corpus_texts, stopwords=STOPWORDS, stemmer=stemmer)
retriever = bm25s.BM25(k1=K1, b=B, method=METHOD) # Default values - can be tuned
retriever.index(corpus_tokens)

# Save Indicies
retriever.save(BM25_DIR)

# save pids
with open(BM25_DIR / "pids.json", "w") as f:
    json.dump(corpus.pid.tolist(), f)

# save config
config = {"text": "html.unescape(title) + ' ' + text", 
          "stopwords": STOPWORDS, "stemmer": STEMMER_LANG, 
          "k1": K1, "b": B, 
          "method": METHOD, "n_docs": len(corpus_texts)}

with open(BM25_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)




Split strings:   0%|          | 0/509315 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/509315 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/509315 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/509315 [00:00<?, ?it/s]

### Queries

In [4]:
# Load Dataset for questions
hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

# Train questions: only the fields needed for Queries (retrieval)
train_q = hotpot_ds["train"].select_columns(["id", "question"]).to_pandas()
train_q = train_q.set_index("id")


SLICES = ["pilot", "tune"]
RETRIEVE_K = 60   # retrieve a few extra so 50 remain after collapsing duplicate titles
KEEP_K = 50
        
pids = corpus.pid.tolist()                              # index row -> pid
pid_to_title = dict(zip(corpus.pid, corpus.title))
        
for slice_name in SLICES:
    with open(SPLITS_DIR / f"{slice_name}.json") as f:
        ids = json.load(f)
    
    questions = train_q.loc[ids, "question"].tolist()
    tokens = bm25s.tokenize(questions, stopwords=STOPWORDS, stemmer=stemmer, show_progress=False)
    rows, scores = retriever.retrieve(tokens, k=RETRIEVE_K, show_progress=False)

    run = {}
    for qid, doc_rows in zip(ids, rows):
        ranked, seen_titles = [], set()
        for row in doc_rows:
            pid = pids[row]
            title = pid_to_title[pid]
            if title in seen_titles:        # whitespace duplicate of a higher-ranked paragraph
                continue
            seen_titles.add(title)
            ranked.append(pid)
            if len(ranked) == KEEP_K:
                break
        run[qid] = ranked


    # Checks
    assert list(run) == ids, "every question, in slice order"
    assert all(len(r) == len(set(r)) <= KEEP_K for r in run.values()), "no repeats, at most 50"
    assert all(len({pid_to_title[p] for p in r}) == len(r) for r in run.values()), "one pid per title"
    print(f"{slice_name}: {len(run)} questions, shortest list {min(map(len, run.values()))}")

    with open(RUNS_DIR / f"bm25_{slice_name}.json", "w") as f:
        json.dump(run, f)



pilot: 200 questions, shortest list 50
tune: 1000 questions, shortest list 50


In [5]:
gold_cols = hotpot_ds["train"].select_columns(["id", "type", "answer", "supporting_facts"]).to_pandas().set_index("id")
qid_to_type = gold_cols["type"].to_dict()

for slice_name in SLICES:
    with open(SPLITS_DIR / f"{slice_name}.json") as f:
        ids = json.load(f)

    # Gold file for this slice (train slices only)
    gold = gold_converter(gold_cols.loc[ids])
    with open(GOLD_DIR / f"{slice_name}_gold.json", "w") as f:
        json.dump(gold, f, indent=2)
    
    with open(RUNS_DIR / f"bm25_{slice_name}.json") as f:
        run = json.load(f)
        
    per_question, summary = score_retrieval(run, gold, qid_to_type, pid_to_title)
    print(f"\n=== BM25, {slice_name} ===")
    print(pd.DataFrame(summary).T.round(2).to_string())


=== BM25, pilot ===
                n  both@2  any@2  mean_gold@2  both@5  any@5  mean_gold@5  both@10  any@10  mean_gold@10  both@20  any@20  mean_gold@20  both@50  any@50  mean_gold@50
overall     200.0    17.0  83.00         1.00   33.00   89.5         1.23    41.00    93.5          1.34    53.50   94.50          1.48    64.00    96.0          1.60
bridge      160.0    15.0  81.25         0.96   30.63   87.5         1.18    35.62    92.5          1.28    46.25   93.75          1.40    56.25    95.0          1.51
comparison   40.0    25.0  90.00         1.15   42.50   97.5         1.40    62.50    97.5          1.60    82.50   97.50          1.80    95.00   100.0          1.95

=== BM25, tune ===
                 n  both@2  any@2  mean_gold@2  both@5  any@5  mean_gold@5  both@10  any@10  mean_gold@10  both@20  any@20  mean_gold@20  both@50  any@50  mean_gold@50
overall     1000.0   20.00  83.80         1.04   37.50  91.70         1.29    48.00   94.20          1.42     56.3   96.50 